In [1]:
%%capture
%matplotlib widget
import numpy as np
from matplotlib import pyplot as plt
import requests
import os, sys
from PIL import Image
!pip install opencv-python
import cv2

## Find Largest Bounding Rectangle and Equal Color Shape Across Images

In [2]:

# Initialize min and max coordinates


mask_found = False

def rotate_image(image, angle):
    image_center = tuple(np.array(image.shape[1::-1]) / 2)
    rot_mat = cv2.getRotationMatrix2D(image_center, angle, 1.0)
    result = cv2.warpAffine(image, rot_mat, image.shape[1::-1], flags=cv2.INTER_LINEAR)
    return result

def calculate_box_size(image, find_silver_ti=False):
    min_x, min_y = np.inf, np.inf
    max_x, max_y = -np.inf, -np.inf
    blurred_img = cv2.GaussianBlur(image, (5, 5), 0)
        
    hsv_img = cv2.cvtColor(blurred_img, cv2.COLOR_RGB2HSV)

    # Threshold the image
    mask = cv2.inRange(hsv_img, (3, 170, 50), (36, 255,255))
    if find_silver_ti:
        mask = cv2.inRange(hsv_img, (0, 0, 150), (180, 50, 220))
        ## Apply morphological operations to remove small noise
        kernel = np.ones((45, 45), np.uint8)
        mask = cv2.erode(mask, kernel, iterations=2)
        mask = cv2.dilate(mask, kernel, iterations=2)

    ## Slice the green
    imask = mask>0
    yellow = np.zeros_like(image, np.uint8)
    yellow[imask] = image[imask]

    # Find contours
    img_gray = cv2.cvtColor(yellow, cv2.COLOR_BGR2GRAY)
    contours, _ = cv2.findContours(img_gray, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Find the bounding rectangle for each contour and update the min/max coordinates
    for cnt in contours:
        x, y, width, height = cv2.boundingRect(cnt)
        min_x = min(min_x, x)
        min_y = min(min_y, y)
        max_x = max(max_x, x + width)
        max_y = max(max_y, y + height)
    
    mask = np.full_like(image, np.nan)

    if min_y == np.inf:
        min_y = 0
    if min_x == np.inf:
        min_x = 0
    if max_y == -np.inf:
        max_y = 1
    if max_x == -np.inf:
        max_x = 1

    # Fill the rectangle in the mask with white (1's) using the min/max coordinates
    mask[min_y:max_y, min_x:max_x] = 1
    masked_img = image * mask

    
    return (yellow,masked_img), ((max_y-min_y),(max_x-min_x))

def find_active_area(file, find_silver_ti=False):
    global data, min_x, min_y, max_x, max_y, mask,  mask_found
   
        
    from PIL import Image
    im = Image.open(file)            
    img = np.array(im)
    final_image, box = calculate_box_size(img, find_silver_ti)
    final_angle = 0
    for angle in np.linspace(-3,3,50):
        img_rotated = rotate_image(img,angle)
        b_img, b = calculate_box_size(img_rotated, find_silver_ti)
        if  b[0]*b[1] < box[0]*box[1]:
            final_image, box, final_angle = b_img,b, angle
    print(final_angle)
    yellow, masked_img = final_image
    imask = yellow != 0
    imask = np.sum(imask, axis=2, dtype=bool)
   
    mean = yellow[imask].mean(axis=(0))
    std = yellow[imask].std(axis=(0))
    sample_width = 2.5
    box_size = sample_width**2*b[0]/b[1]
    active_area = box_size * cv2.countNonZero(cv2.cvtColor(yellow, cv2.COLOR_BGR2GRAY))/cv2.countNonZero(cv2.cvtColor(masked_img, cv2.COLOR_BGR2GRAY))
    active_area2 = 9 * cv2.countNonZero(cv2.cvtColor(yellow, cv2.COLOR_BGR2GRAY))/cv2.countNonZero(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY))
    
    plt.close()
    
    plt.figure(figsize=(12,5))

    plt.subplot(131)
    plt.title(f"Active area: {round(active_area,3)}, {round(active_area2,3)}", fontsize=8)
    plt.imshow(yellow)
    plt.colorbar()

    plt.subplot(132)
    plt.title(f"Masked Sample \n(Mean: {mean.round(0)}, Std: {std.round(0)})", fontsize=8)
    plt.imshow(masked_img, vmin=0)
    plt.colorbar()
    
    plt.subplot(133)
    plt.title(f"Image", fontsize=8)
    plt.imshow(img, vmin=0)


    plt.show()

    mask_found = True 



In [3]:
import ipywidgets as widgets
from IPython.display import display
import sys
sys.path.insert(1, '../../python-scripts-c6fxKDJrSsWp1xCxON1Y7g')
from api_calls import get_all_entries_of_type
url = "https://nomad-hzb-ce.de/nomad-oasis/api/v1"
import os
token = os.environ['NOMAD_CLIENT_ACCESS_TOKEN']

def get_nome_scanner_photos(my_ids_only=False):
    if my_ids_only:
        owner = 'user'
    else:
        owner = 'visible'
    data = get_all_entries_of_type(url, token, entry_type='CE_NOME_Measurement')
    return data

def get_dropdown_text_options(data):
    return [('-- no ID selected --', {})] + [(f"{item.get('entry_name')} | {item.get('upload_name')}", item) for item in data if item.get('entry_name', '').endswith('jpg') or item.get('entry_name', '').endswith('tif')]

def on_checkbox_change(change):
    my_ids_only = change['new']
    photos = get_nome_scanner_photos(my_ids_only)    
    photo_dropdown.options = get_dropdown_text_options(photos)

def on_id_dropdown_change(change):
    out.clear_output()
    with out:
        upload_name = photo_dropdown.value.get('upload_name', '').lower().replace(' ', '-')
        separator = '-' if upload_name != '' else ''
        file_name = f'../../{upload_name}{separator}{photo_dropdown.value.get('upload_id', '')}/{photo_dropdown.value.get('entry_name', '')}'
        print(file_name)
        find_active_area(file_name, find_silver_ti_checkbox.value)

own_ids_checkbox = widgets.Checkbox(
    value=False,
    description='just show my own scanner photos',
    layout=widgets.Layout(width='600px'),
    style={'description_width': 'initial'},
)

find_silver_ti_checkbox = widgets.Checkbox(
    value=False,
    description='find silver colored areas (e.g. for Ti)',
    layout=widgets.Layout(width='600px'),
    style={'description_width': 'initial'},
)

initial_photos = get_nome_scanner_photos(own_ids_checkbox.value)
photo_dropdown = widgets.Dropdown(
    options=get_dropdown_text_options(initial_photos),
    description='NOME Photos:',
    layout=widgets.Layout(width='600px'),
    style={'description_width': 'initial'},
)

own_ids_checkbox.observe(on_checkbox_change, names='value')
find_silver_ti_checkbox.observe(on_id_dropdown_change, names='value')
photo_dropdown.observe(on_id_dropdown_change, names='value')

#display(own_ids_checkbox, find_silver_ti_checkbox, photo_dropdown)
display(find_silver_ti_checkbox, photo_dropdown)

out = widgets.Output()
display(out)

Checkbox(value=False, description='find silver colored areas (e.g. for Ti)', layout=Layout(width='600px'), sty…

Dropdown(description='NOME Photos:', layout=Layout(width='600px'), options=(('-- no ID selected --', {}), ('CE…

Output()

In [4]:
#file_name = "../../ce-nome_mazo_240709_0004_tio2-jS1oznNDQqSrqjCQcmTURA/CE-NOME_MaZo_240709_0004_TiO2.tif"
#file_name = "../ce-nome_somu_240716_0001_tio2-ihAmqhgiQ5aJAnZgnNQnDw/CE-NOME_SoMu_240716_0001_TiO2.tif"
#find_active_area(file_name, find_silver_ti=True)